In [54]:
import numpy as np
from numpy.typing import NDArray

In [ ]:
class WeightMatrix:
    _corpora: NDArray
    _dataset: NDArray[np.str_] | None = None
    _embeddings: NDArray[np.float64] | None = None

    wti: dict # word to index

    # Corpora filename could be a hash to enable comparing versions of the file and identify changes
    paths: dict = {
        "corpora": "./corpora/corpora.csv",
        "dataset": "dataset.npy",
        "embeddings": "embeddings.npy",
    }

    def __init__(self, pathkey='corpora') -> None:
        self._corpora = self._load_corpora(pathkey)
        self._vocabulary = self._load_vocabulary()
        self._embeddings = self._load_embeddings()

        self.wti = {}

    def _load_corpora(self, pathkey='corpora') -> NDArray:
        try:
            self._corpora = np.loadtxt(
                self.paths[pathkey], dtype=str, delimiter=";", skiprows=1
            )

            return self._corpora
        except Exception as e:
            print(e)
            raise Exception("corpora not found")

    def _load_vocabulary(self) -> NDArray | None:
        try:
            dataset = np.load(self.paths["dataset"], "r")
        except Exception as e:
            print(e, " NOT CREATED YET")
            return None

        return dataset

    def _load_embeddings(self) -> NDArray | None:
        try:
            embeddings = np.load(self.paths["embeddings"], "r")
        except Exception as e:
            print(e, " – NOT CREATED YET")
            return None

        return embeddings
    
    def clean_token(self, t: str, chars: list[str]):
        if len(chars) == 0:
            return t 
        stopword_idx =  t.find(chars[-1])
        if stopword_idx < 0:
            chars.pop()
        else:
            t = ''.join(list(t).pop(stopword_idx))

        return self.clean_token(t, chars)
         
    
    @property
    def corpora(self) -> NDArray:
        return self._corpora[:, -1]

    @property
    def vocabulary(self) -> NDArray:
        if self._vocabulary is None:
            raise Exception("You must call `build_and_retrieve` before calling dataset")

        return self._vocabulary

    @property
    def embeddings(self) -> NDArray:
        if self._embeddings is None:
            raise Exception()

        return self._embeddings

    # --- Public API --- ?

    def build_and_retrieve(self):
        self._build_vocabulary()._build_matrix()

    # --- Private functions ---

    def _build_vocabulary(self) -> "WeightMatrix":

        print(self.corpora)
        self._vocabulary = np.array(
            sorted(
                list(
                    set(
                        [self.clean_token(word, [".", ":", "!"]) for sentence in self.corpora for word in sentence.split()],
                    )
                ),
                key=lambda x: x,
            )
        )

        np.save("vocabulary.npy", self._vocabulary)
        return self

    def _build_matrix(self):
        if self._vocabulary is None:
            raise Exception()

        rng = np.random.default_rng()
        embeddings = []
        for i, w in enumerate(self._vocabulary):
            # Um embedding está sendo criado randomicamente para a palavra w numa posição i específica no array
            # de embeddings AND
            # criamos uma tabela para procurar o índice dessa palavra em embeddings (palavra -> index)
            embeddings.append(rng.standard_normal(3))
            self.wti[w] = i

        # self._embeddings = np.array([rng.standard_normal(3) for _ in self._vocabulary])
        self._embeddings = np.array(embeddings)
        np.save("embeddings.npy", self._embeddings)

In [57]:
def context_mapping(i: int) -> NDArray:
    match i:
        case 0:
            return np.array([1, 2, 3, 4])
        case 1:
            return np.array([-1, 1, 2, 3])
        case -2:
            return np.array([-3, -2, -1, 1]) 
        case -1:
            return np.array([-4, -3, -2, -1])
        case _:
            return np.array([-2, -1, 1, 2])


wm = WeightMatrix()
wm.build_and_retrieve()

vocabulary = wm.vocabulary
embeddings = wm.embeddings
corpora: NDArray = wm.corpora

data = []
target = []
for sentence in corpora:
    # Remove dots from tokens (words)
    splitted_sentence = np.array(sentence.split())
    for i, w in enumerate(splitted_sentence):
        key = i if i not in [len(splitted_sentence) -2, len(splitted_sentence) -1] else -(len(splitted_sentence) - i)
        # training_chunk.append(splitted_sentence[context_mapping(key) + i]) 
        data.append(splitted_sentence[context_mapping(key) + i])
        target.append(w)
    
    # training_chunk = np.array(training_chunk)
    # np.vstack((dataset, training_chunk))


data = np.array(data)
target = np.array(target)
print(data.shape)

for d, t in zip(data[:10], target[:10]):
    print(f"{t}: {d}")

# for tr, ta in zip(training_chunk, target_chunk):
#     print(tr, ta)




[Errno 2] No such file or directory: 'dataset.npy'  NOT CREATED YET
[Errno 2] No such file or directory: 'embeddings.npy'  – NOT CREATED YET


RecursionError: maximum recursion depth exceeded